In [1]:
import automuse as am
import automuse.chord as chord
import automuse.modes as modes
from automuse.scale import scale
from automuse import note_i2s
from automuse import note_s2i
from automuse import interval_s2i
from automuse import interval_i2s
from automuse import name_interval
from automuse import name_intervals
from automuse import reach
from automuse import same_class

# An Introduction to Music Theory

This notebook covers the basics of music theory, namely intervals, modes, scales, and chords.

## Intervals

An **interval** is the musical distance between two notes. An interval between two notes is **harmonic** if these notes are played at the same time; otherwise, the interval is **melodic**. These intervals are important -- all music is the combination of melodies and harmonies.

The **half-step**, (also **semitone**) is the smallest apartness commonly used in Western music. A **whole-step** equals two half-steps.

In integer notation, an interval is denoted by a simple integer. In text however, intervals are often communicated by name. The following sections explain how these names are constructed.

### Generic Intervals

**Generic intervals** measure the difference between the staff positions of two notes. In practice, this measure ignores accidentals: $\mathrm{C}\text{--}\mathrm{D}$ are one genetic step apart, but so are $\mathrm{C}\text{--}\mathrm{D}\#$. This is because $\mathrm{D}$ and $\mathrm{D}\#$ are on the same staff.

Generic intervals, like people, have names. 
For example, $\mathrm{C}\text{--}\mathrm{D}$ are a second apart. The following table lists these names.

|Step difference|Name of Interval|
|-|-|
| 0 | First / Prime / Unison |
| 1 | Second  |
| 2 | Third   |
| 3 | Fourth  |
| 4 | Fifth   |
| 5 | Sixth   |
| 6 | Seven   |
| 7 | Eight   |

### Specific Intervals

**Specific intervals** are measured on both staff and half steps. For example, recall that $\mathrm{C}\text{--}\mathrm{D}$ are a generic second apart. Because they are also 2 half steps apart, they are a _major second_ apart in specific intervals. $\mathrm{B}\text{--}\mathrm{C}$ on the other hand are a _minor second_ apart because they are a genetic second apart while being just one half step apart.

The terms "major" and "minor" refer to the interval's **quality**. Only the 2<sup>nd</sup>s, 3<sup>rd</sup>s, 4<sup>th</sup>s, 6<sup>th</sup>s, and 7<sup>th</sup>s can be **major interval**s. The rest (1<sup>st</sup>s, 4<sup>th</sup>s, 5<sup>th</sup>s, and 8<sup>th</sup>s) are **perfect interval**s instead.

A **minor interval** has 1 fewer half step than a major interval. An **augmented interval** has one more than a major interval. An **augmented interval** has one more half step than a perfect interval. A **diminished interval** has one less half step. Minor intervals can be diminished by subtracting yet another half-step. The following figure illustrates these relations:

![image](../../media/music_steps.svg)

Some examples follow:

|Apartness|Name|
|-|-|
| 0 | Prime          |
| 1 | Minor Second   |
| 3 | Minor Third    |
| 5 | Perfect Fourth |
| 7 | Perfect Fifth  |
| 9 | Major Sixth    |
| 11 | Major Seventh |
| 12 | Perfect Eighth (Perfect octave) |

To add, **compound intervals** are larger than an octave. These intervals are named by adding 7 to the generic interval.

Following these rules, there exists a injective mapping between names and integers (difference in semitones). A set of utilities in AutoMuse work with this fact:

* `INTERVALS` maps every name to an integer.

* `name_interval` takes two notes and returns a name for their interval.

In [2]:
print(f"Augmented 2 is {am.INTERVALS["augmented 2"]} in semitones.\n")
_test_pairs: list[tuple[str, str]] = [
    ('C#', 'D'),
    ('C', 'D'),
    ('C', 'D#'),
    ('B', 'C'),
]

for __from, to in _test_pairs:
    print(f"Interval {__from}"
          f" --> {to}: "
          f" {name_interval(__from, to)}")

_TEST_CHORD: list[str] = ["C", "E", "G"]
print(f"\nIntervals in {name_intervals(_TEST_CHORD)}")

Augmented 2 is 3 in semitones.

Interval C# --> D:  minor 2
Interval C --> D:  major 2
Interval C --> D#:  augmented 2
Interval B --> C:  minor 2

Intervals in ['major 3', 'minor 3']


#### Intervals Cheat Sheet

The following generated cheat sheet gives possible names for an interval in half steps. To find out the actual name, choose one that corresponds to the general interval.

In [3]:
persephone: list[tuple[int, str]]\
       = sorted([(b, a) for a, b
                 in am.INTERVALS.items()])

import tabulate
tabulate.tabulate((p := persephone,
                  [(*x, *y) for (x, y)
                            in zip(p[:int(len(p)/2)],
                                   p[int(len(p)/2):])])[1],

                  headers=["Half-Step Difference",
                           "Name"]*2,
                  tablefmt="html")

Half-Step Difference,Name,Half-Step Difference,Name
0,diminished 2,7,diminished 6
0,prime 1,7,perfect 5
1,augmented 1,8,augmented 5
1,minor 2,8,minor 6
2,diminished 3,9,diminished 7
2,major 2,9,major 6
3,augmented 2,10,augmented 6
3,minor 3,10,minor 7
4,diminished 4,11,diminished 1
4,major 3,11,diminished 8


The table shows an interesting fact: that the same semitone difference can have different names, depending on the generic interval. This is an instance of **enharmonic equivalence** in music where two things sound the same yet write differently.

### Reaching by Intervals

The `reach` function "reaches up" from a given note by either an integer or an interval name.

To demonstrate the correctness of `reach`, assuming that `name_interval` is correct, let's  reach from every note to every other note by the name of the interval between them.

In [4]:
for name_x in am.NOTE_NAMES:
    for name_y in am.NOTE_NAMES:
        same_class(reach(name_x, name_interval(name_x, name_y)), name_y)


### Intervals in the Circle of Fifth

Without going into details, the [Circle of Fifths](./circle_of_fifths.ipynb) arranges musical notes in two rings.

Take any note $N_1$ on the exterior ring and its counterpart $N_2$ on the interior ring. The major scale in $N_1$ has the same notes as the minor scale in $N_2$. Also, $N_2$ is a major 6th from $N_1$.

Let's see if AutoMuse respect these relations:

In [5]:
from automuse import same_class, name_interval

CIRCLE_OF_FIFTHS: list[tuple[str, str]]\
    = [("C", "A"),
       ("G", "E"),
       ("D", "B"),
       ("A", "F#"),
       ("E", "C#"),
       ("B", "G#"),
       ("F#", "D#"),
       ("C#", "A#"),
       ("G#", "F"),
       ("D#", "C"),
       ("A#", "G")]

for major_key, minor_key in CIRCLE_OF_FIFTHS:
    assert same_class(
        scale(major_key, modes.MAJOR),
        scale(minor_key, modes.MINOR),)

    interval_name: str = name_interval(major_key, minor_key)

    print(f"Interval from {major_key} to {minor_key}"
          f" is {name_interval(major_key, minor_key)},"
          f" or {am.INTERVALS[interval_name]} half steps.")

Interval from C to A is major 6, or 9 half steps.
Interval from G to E is major 6, or 9 half steps.
Interval from D to B is major 6, or 9 half steps.
Interval from A to F# is major 6, or 9 half steps.
Interval from E to C# is major 6, or 9 half steps.
Interval from B to G# is major 6, or 9 half steps.
Interval from F# to D# is major 6, or 9 half steps.
Interval from C# to A# is major 6, or 9 half steps.
Interval from G# to F is diminished 7, or 9 half steps.
Interval from D# to C is diminished 7, or 9 half steps.
Interval from A# to G is diminished 7, or 9 half steps.


## Scales

A **scale** is an ordered sequence of notes. In western music, a scale (particularly a **diatonic scale**) is constructed by counting notes from a starting note. This starting note is its **home note** (or _**tonic**_); the pattern of counting is either its **key** (if the pattern is major or minor) or its **mode** (otherwise). These inconsistencies are due to historical reasons.

Mathematically, both keys and modes are patterns of intervals. These patterns can be found in `automuse.modes`.

Parts of this section are based on [MusicTheory.net](www.musictheory.net) and [_Play Guitar in 14 Days_](https://troynelsonmusic.com/).

### Aside: Major and Minor

Scales, for example the **major scale**s and **minor scale**s, are constructed from a pattern of intervals. In this context, the word **key** can communicate two things: (a) if a scale is major or minor (e.g. "the scale is in major key"), or (b) the tonic (e.g. "the scale is in the key of A"). Better not think too hard about it.

Here, you see another use of the terms "major" and "minor". These two terms can apply to a great many things, but one rule remains consistent: a _sound_, be it a chord, a scale, or a juxtaposition of two pitches, is major if it is stable or bright; on the other hand, sounds that are not are minor.

### Aside: Scale Degrees

The **scale degree** of a note is its position on a scale, up from the tonic. The $i^\mathrm{th}$ degree is denoted as $\hat{i}$.

For a heptatonic scale, these degrees are named:

| Position | Name                    |
| -------- | ----------------------- |
| 8 (out of scale) | Tonic (again)   |
| 7        | Leading Tone / Subtonic |
| 6        | Submediant              |
| 5        | Dominant                |
| 4        | Subdominant             |
| 3        | Mediant                 |
| 2        | Supertonic              |
| 1        | Tonic                   |

Note that the submediant ($\hat{6}$) does not lead into the mediant ($\hat{3}$); rather, it is the "mediant" of the dominant ($\hat{5}$) and the subtonic ($\hat{7}$).

Also, the $\hat{7}$ note can have two names: If it is one half step below the tonic, then it is the leading tone; if it is one whole step below the tonic, then it is the subtonic. These names are often used interchangeably.

### Constructing Scales

To construct a scale from a tonic and a mode, start counting semitones from the tonic according to the mode. The `scale.scale` function constructs scales in this manner.

At minimum, the function requires a tonic and a mode.

In [6]:
scale("C", modes.MAJOR)

['C0', 'D0', 'E0', 'F0', 'G0', 'A0', 'B0']

By default, scales are represented by intervals between notes. The isomorphic **integer notation** denotes each note by its offset from the root. This notation should be intuitive for people with a background in music.

In this notation, the major chord becomes $[0, 4, 7]$, since a major 3 is 5 semitones and a perfect 5 is 7 semitones. To exclude the root, simply remove 0 from the pattern. This is not possible in the intervals notation.

Setting `use_offsets=True` lets `scale` accept offsets instead of intervals.

In [7]:
scale("C", [0, 2, 4, 5, 7, 9, 11], use_offsets=True)

['C0', 'D0', 'E0', 'F0', 'G0', 'A0', 'B0']

These notations are isomorphic, and a set of functions `offsets_to_intervals` and `intervals_to_offsets` map between them.

In [8]:
modes.offsets_to_intervals(
    modes.intervals_to_offsets(modes.MAJOR)) ==\
    modes.MAJOR

True

### Pentatonic Scales

A pentatonic scale is a scale with five tones instead of seven. To construct a pentatonic scale from a major heptatonic scale, take items at $\{1, 2, 3, 5, 6\}$ (assuming 1-based indexing). Constructing pentatonic minor scales is similar, but uses $\{1, 3, 4, 5, 7\}$ instead.

The `pentatonic_major` and `pentatonic_minor` functions construct pentatonic "modes", which can then be used to construct pentatonic scales. These functions can also scales from other modes.

You can also construct scales from two pre-made modes, `MAJOR_PENTATONIC` and `MINOR_PENTATONIC`.

In [9]:
assert modes.MAJOR_PENTATONIC == modes.pentatonic_major(modes.MAJOR)
assert modes.MINOR_PENTATONIC == modes.pentatonic_minor(modes.MINOR)

### Blues Scales

A blues minor scale is the pentatonic scale with an extra flat $\hat{5}$. For example, whereas the A pentatonic scale is $[\text{A}, \text{C}, \text{D}, \text{E}, \text{G}]$, the A blues scale is $[\text{A}, \text{C}, \text{D}, \text{D}\#, \text{E}, \text{G}]$. A blues major scale gets a flat third instead.

The `blues_major` and `blues_minor` functions construct blue modes. As usual, pre-made modes for major and minor are also available.

In [10]:
assert modes.MAJOR_BLUES == modes.blues_major(modes.MAJOR)
assert modes.MINOR_BLUES == modes.blues_minor(modes.MINOR)

assert same_class(
    scale('C', modes.blues_major(modes.MAJOR)),
    ['C', 'D', 'D#', 'E', 'G', 'A']
)

assert same_class(
    scale('A', modes.blues_minor(modes.MINOR)),
    ['A', 'C', 'D', 'D#', 'E', 'G']
)

### Harmonic and Melodic Minors

I see your harmonic minor and raise you a melodic minor.

A harmonic minor has a raised $\hat{7}$. A melodic minor has both its $\hat{6}$ and $\hat{7}$ raised.

The `harmonic_minor` and `melodic_minor` functions construct these modes.

In [11]:
assert modes.MINOR_HARMONIC == modes.harmonic_minor(modes.MINOR)
assert modes.MINOR_MELODIC == modes.melodic_minor(modes.MINOR)

assert same_class(
    scale("A", modes.melodic_minor(modes.MINOR)),
    ['A', 'B', 'C', 'D', 'E', 'F#', 'G#'])

assert same_class(
    scale("A", modes.harmonic_minor(modes.MINOR)),
    ['A', 'B', 'C', 'D', 'E', 'F', 'G#'])

### Augmented and Diminished Scales

There are two kinds of diminished scales: the
typical whole-half diminished scale (of 
intervals $[2, 1, 2, 1, ...]$) and the half-whole diminished scale (of intervals $[1, 2, 1, 2, ...]$).


In [12]:
from automuse.modes import DIMINISHED_WHOLE_HALF
from automuse.modes import DIMINISHED_HALF_WHOLE
from automuse.modes import AUGMENTED

In [13]:
assert same_class(
    scale("A", modes.DIMINISHED_HALF_WHOLE),
    ['A', 'A#', 'C', 'C#', 'D#', 'E', 'F#', 'G'])

## Chords

Simply put, a **chord** is a collection of tones. There are several ways to construct harmonically interesting chords. Let's start with the triad:

As you know, a triad is a triple of topological spaces $\{P, A, B\};~A,B\prec P$ where $P=\mathrm{int}(A)\cup\mathrm{int}(B)$.

To construct a triad, select a scale then pick its $\hat{1}$, $\hat{3}$, and $\hat{5}$. The $\hat{1}$ is the root of the chord and remains so even if it is, after the triad is transformed, no longer the lowest note.

### Constructing a Triad

AutoMuse implements two ways to construct chords. Let us begin with the easiest: `chord.chord`. This function implements all you need to construct a chord from specifications.

To see how it works, let's construct this abomination of a C chord:

In [14]:
chord.chord(
    tonic="C4",
    mode=modes.IONIAN,
    order=0,
    add=[6, 8, 10],
    sus=[0],
    sharp=[2],
    flat=[4],
    raw_offsets=["augmented 1"]
)

['E4', 'F4', 'G4', 'A4', 'A#4', 'B4', 'D5', 'F5']

Alternatively, you can construct chords with `scale.scale`. `modes` has several options that specify chords, instead of scales:

In [15]:
scale("C4", modes.TRIAD_MAJOR)

['C4', 'E4', 'G4']

### Seventh Chords

A seventh chord combines a triad with a seventh. The `chord.seventh` function construct seventh chords.

You can simply pick $\hat{1}$, $\hat{3}$, $\hat{5}$, and $\hat{7}$ from the same scale, or specify the quality for the $\hat{7}$:

In [16]:
print(chord.seventh("C", modes.LOCRIAN))
print(chord.seventh("C", modes.LYDIAN))
print(chord.seventh("C", modes.LYDIAN, type="diminished"))

['C0', 'D#0', 'F#0', 'A#0']
['C0', 'E0', 'G0', 'B0']
['C0', 'E0', 'G0', 'A0']


### Diatonic Triads

Every major and minor scale have seven **diatonic triads**, which are formed from notes on that scale.

The first triad uses the 1<sup>st</sup>, 3<sup>rd</sup>, and 5<sup>th</sup> notes counting up from the root (the root itself being the 1<sup>st</sup>). The n<sup>th</sup> triad instead counts from the n<sup>th</sup> note instead.

Both `chord.chord` and `chord.seventh` can construct chords of the given `order`. To make $\text{ii}^7$, write:

In [17]:
chord.seventh("C", modes.MAJOR, order=1)

['D0', 'F0', 'A0', 'C1']

### Neapolitan Chords

To construct a Neapolitan chord, construct a major triad (1, 3, 5) starting with a flat second of the major scale. We can see that it is equivalent to the second triad on a Phrygian scale:

In [18]:
print(chord.chord("A", modes.PHRYGIAN, order=1))
print(chord.neapolitan_chord("A", modes.MAJOR))

['A#0', 'D1', 'F1']
['A#0', 'D1', 'F1']


Only because it is possible ... you can construct a Neapolitan chord on the Phrygian scale.

In [19]:
print(chord.neapolitan_chord("A", modes.PHRYGIAN))

['A0', 'C#1', 'E1']


### Inverting Triads

Like inverting intervals, inverting a triad moves the lowest note up an octave. 

When the **bass note**, the lowest note in the triad, is its root, the triad is in **root position**. Inverting the triad once moves it into **first inversion**; inverting again moves it to the **second inversion**. After inversion, the chord's base note remains the same, but its bass note becomes the new lowest note.

Alternatively, the degree of inversion can be denoted by its bass note. For example, the F major triad is $[\text{F}, \text{A}, \text{C}]$; its first inversion $[\text{A}, \text{C}, \text{F}]$ can be denoted by $\text{F}/\text{A}$. This notation is called a **slash chord**.

The `automuse.transform` module contains several such transforms. It supports both notations by `invert_chord_by` and `invert_triad_to`.

In [20]:
from automuse.transforms import invert_chord_to, invert_chord_by
_c_4_triad = chord.chord("C4", modes.MAJOR)
print(invert_chord_to(_c_4_triad, _c_4_triad[1]))
print(invert_chord_by(_c_4_triad, 1))

['E4', 'G4', 'C5']
['E4', 'G4', 'C5']
